In [1]:
from utils import parse_benchmark_to_dataframe
from utils import filter_by_masks
from utils import df_combined_to_markdown
from utils import df_combined_to_csv
from static import *

In [2]:
df, df_mean, df_sd, df_rsd, df_combined, df_flent = parse_benchmark_to_dataframe(project_dir / "results")

In [3]:
# delay, jitter, loss, cpu_freq, core_count
masks = [
    # 0. Delay in isolation
    [(0, 0, 0, 4.0, 4), (50, 0, 0, 4.0, 4), (100, 0, 0, 4.0, 4), (300, 0, 0, 4.0, 4)],
    # 1. Real network conditions
    [(0, 0, 0, 4.0, 4), (100, 20, 0, 4.0, 4), (100, 80, 0, 4.0, 4)],
    # 2. Packet loss in isolation
    [(0, 0, 0, 4.0, 4), (0, 0, 0.1, 4.0, 4), (0, 0, 1.0, 4.0, 4), (0, 0, 3.0, 4.0, 4)],
    # 3. Lower tier / low-end machines
    [(0, 0, 0, 4.0, 4), (0, 0, 0, 2.0, 1), (0, 0, 0, 2.0, 2)],
    # 4. Standard conditions
    [(0, 0, 0, 4.0, 4), (40, 10, 0.5, 4.0, 4)],
    # 5. Extreme out-of-order delivery
    [(0, 0, 0, 4.0, 4), (100, 80, 0, 4.0, 4)],
]

# Pipeline configuration: (mask, sort_columns, output_filename_base)
pipelines = [
    (masks[0], ["delay", "vpn"], "delay"),
    (masks[1], ["delay", "jitter", "vpn"], "real"),
    (masks[2], ["loss", "vpn"], "loss"),
    (masks[3], ["cpu_freq", "core_count", "vpn"], "low_end"),
    (masks[4], ["delay", "jitter", "loss", "vpn"], "standard"),
    (masks[5], ["delay", "jitter", "vpn"], "extreme_ooo"),
]

output_dir = project_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Run pipeline
for mask, sort_cols, filename in pipelines:
    sdf = filter_by_masks(df_combined, mask).sort_values(by=sort_cols, ascending=False)

    csv_data = df_combined_to_csv(sdf)
    md_data = df_combined_to_markdown(sdf, sd_thresholds=thresholds, rsd_threshold=0.15)

    (output_dir / f"{filename}.csv").write_text(csv_data, encoding="utf-8")
    (output_dir / f"{filename}.md").write_text(md_data, encoding="utf-8")

# Dump combined views
combined_csv = df_combined_to_csv(df_combined)
combined_md = df_combined_to_markdown(df_combined, sd_thresholds=thresholds, rsd_threshold=0.15)
(output_dir / "combined.csv").write_text(combined_csv, encoding="utf-8")
(output_dir / "combined.md").write_text(combined_md, encoding="utf-8")

# Dump raw views
raw_md = df.round(2).sort_values(by=key_cols, ascending=False).to_markdown()
(output_dir / "raw.md").write_text(raw_md, encoding="utf-8")

58343